In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

from jColor.Color import Color
from jColor.Image import Image

In [ ]:
SAMPLE = 'M1'
image_path =  f'imgs/{SAMPLE}.jpg'

jade_image = Image(str(image_path))
normalized_rgb, white_reference, foreground_mask = jade_image.RGB_Normalization(bg_remover=True)
foreground = foreground_mask.astype(bool)

print(f'Sample: {SAMPLE}')
print(f'RGB - White reference: {np.round(white_reference, 2)}')
print(f'Number of sample pixels: {np.count_nonzero(foreground):,}')

plt.figure(figsize=(7, 6), dpi = 300)
plt.imshow(normalized_rgb)
plt.title(f'{SAMPLE}')
plt.axis('off')
plt.show()

In [ ]:
lab_image = Color(normalized_rgb).Lab
L_image, a_image, b_image = cv2.split(lab_image)

N_CLUSTERS = 3
GROUP_NAMES = np.array(['A', 'B', 'C'])
GROUP_COLORS = ['black', 'red', 'darkblue']

lab_pixels = lab_image[foreground]
kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    init='k-means++',
    n_init=10,
    random_state=0,
)
original_labels = kmeans.fit_predict(lab_pixels)

cluster_order = np.argsort(kmeans.cluster_centers_[:, 0])[::-1]
old_to_new = np.empty(N_CLUSTERS, dtype=int)
old_to_new[cluster_order] = np.arange(N_CLUSTERS)
ordered_labels = old_to_new[original_labels]
ordered_centers = kmeans.cluster_centers_[cluster_order]

segmentation_map = np.full(L_image.shape, N_CLUSTERS, dtype=np.uint8)
segmentation_map[foreground] = ordered_labels
map_colors = GROUP_COLORS + ['white']
segmentation_cmap = ListedColormap(map_colors)

centers_table = pd.DataFrame(
    ordered_centers, index=GROUP_NAMES, columns=['L*', 'a*', 'b*']
)
display(centers_table.round(3))

plt.figure(figsize=(7, 6))
plt.imshow(
    segmentation_map, cmap=segmentation_cmap,
    vmin=-0.5, vmax=N_CLUSTERS + 0.5, interpolation='nearest'
)
plt.title(f'{SAMPLE}')
plt.axis('off')
plt.show()

In [ ]:
total_pixels = np.count_nonzero(foreground)
statistics = []

fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(15, 5), constrained_layout=True)

for group_id, (group_name, group_color) in enumerate(zip(GROUP_NAMES, GROUP_COLORS)):
    group_mask = segmentation_map == group_id
    group_lab = lab_image[group_mask]
    pixel_count = group_lab.shape[0]
    percentage = 100 * pixel_count / total_pixels

    for channel_id, channel_name in enumerate(['L*', 'a*', 'b*']):
        values = group_lab[:, channel_id]
        statistics.append({
            'sample': SAMPLE,
            'group': group_name,
            'channel': channel_name,
            'mean': values.mean(),
            'standard_deviation': values.std(ddof=0),
            'minimum': values.min(),
            'maximum': values.max(),
            'variance': values.var(ddof=0),
            'median': np.median(values),
            'standard_error': values.std(ddof=0) / np.sqrt(pixel_count),
            'percentage': percentage,
            'pixel_count': pixel_count,
        })

    binary_cmap = ListedColormap(['white', group_color])
    axes[group_id].imshow(group_mask, cmap=binary_cmap, vmin=0, vmax=1, interpolation='nearest')
    axes[group_id].set_title(f'Group {group_name}: {percentage:.2f}%')
    axes[group_id].axis('off')

plt.show()

statistics_table = pd.DataFrame(statistics)
print(f'Sum of percentages: {statistics_table.drop_duplicates("group")["percentage"].sum():.2f}%')
display(statistics_table.round(4))